# Aggregate elastic preds to find best hyper-params w/ MSE and r^2

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from tqdm.auto import tqdm

BASE_SAVE_DIR = BASE_PATH / "initial_preds"

In [ ]:
save_dir_lr = BASE_SAVE_DIR / "agg_type" / "impact_type" / "model_name" / "seg_type"

save_dir_shot = (
    BASE_SAVE_DIR
    / f"0_shot_{"pred_type"}"
    / "impact_type"
    / "model_name"
    / f"desc_{"desc_type"}"
)

In [ ]:
save_dir_lr.relative_to(BASE_PATH).parts

In [ ]:
save_dir_shot.relative_to(BASE_PATH).parts

In [ ]:
save_dir_lr[6]

## Get best params via CV

Optimize for one of:
- r2
- mse

In [ ]:
OPTIMIZE = "r2"
# OPTIMIZE = "mse"
print(f"Optimizng for: {OPTIMIZE}")
L1_LIST = [0.1, 0.1, 0.1]
ALPHA_LIST = [0.001, 0.01, 0.1]

In [ ]:
def get_best_params_score(import_dir, show_progress=False, optimize="mse"):
    if optimize not in ["mse", "r2"]:
        raise ValueError(f"optimize param must be one of ['mse', 'r2'], got {optimize}")
    best_params_score_dict = {
        "avg_mse": 9999999,
        "stdev_mse": 9999999,
        "avg_r2": 0,
        "stdev_r2": 9999999,
        "l1_1": "None",
        "alpha_1": "None",
        "l1_2": "None",
        "alpha_2": "None",
        "l1_3": "None",
        "alpha_3": "None",
    }
    total_files = sum(
        1
        for alpha_dir in import_dir.iterdir()
        for l1_dir in alpha_dir.iterdir()
        for repeat_dir in l1_dir.iterdir()
        for _ in repeat_dir.iterdir()
    )
    with tqdm(
        total=total_files,
        desc="Scoring params",
        leave=False,  # disappears when done
        dynamic_ncols=True,
        disable=not show_progress,
    ) as pbar:
        for alpha_dir in import_dir.iterdir():
            alpha_list = alpha_dir.name.split("_")
            alpha_1 = alpha_list[1]
            alpha_2 = alpha_list[2]
            alpha_3 = alpha_list[3]
            if (
                (alpha_1 != ALPHA_LIST[0])
                or (alpha_2 != ALPHA_LIST[1])
                or (alpha_3 != ALPHA_LIST[2])
            ):
                continue

            for l1_dir in alpha_dir.iterdir():
                l1_list = l1_dir.name.split("_")
                l1_1 = l1_list[1]
                l1_2 = l1_list[2]
                l1_3 = l1_list[3]
                if (l1_1 != L1_LIST[0]) or (l1_2 != L1_LIST[1]) or (l1_3 != L1_LIST[2]):
                    continue
                mse_list = []
                r2_list = []
                for repeat_dir in l1_dir.iterdir():
                    for split_file in repeat_dir.iterdir():
                        df_import = pd.read_parquet(split_file)
                        y_true = df_import["true"]
                        y_pred = df_import["pred_stage3_total"]
                        mse = mean_squared_error(y_true=y_true, y_pred=y_pred)
                        r2 = r2_score(y_true=y_true, y_pred=y_pred)
                        mse_list.append(mse)
                        r2_list.append(r2)
                        pbar.update(1)
                # get avg mse across folds/repeats for l1-alpha combo
                avg_mse = np.mean(mse_list)
                avg_r2 = np.mean(r2_list)
                mse_condition = (optimize == "mse") and (
                    avg_mse < best_params_score_dict["avg_mse"]
                )
                r2_condition = (optimize == "r2") and (
                    avg_r2 > best_params_score_dict["avg_r2"]
                )

                if mse_condition or r2_condition:
                    stdev_mse = np.std(mse_list)
                    stdev_r2 = np.std(r2_list)
                    best_params_score_dict = {
                        "avg_mse": round(avg_mse, 3),
                        "stdev_mse": round(stdev_mse, 3),
                        "avg_r2": round(avg_r2, 4),
                        "stdev_r2": round(stdev_r2, 4),
                        "l1_1": l1_1,
                        "alpha_1": alpha_1,
                        "l1_2": l1_2,
                        "alpha_2": alpha_2,
                        "l1_3": l1_3,
                        "alpha_3": alpha_3,
                    }
    return best_params_score_dict

Full

In [ ]:
df_rows = []
for agg_dir in BASE_SAVE_DIR.iterdir():
    agg_type = agg_dir.name
    if agg_dir.is_file():
        continue
    for impact_dir in agg_dir.iterdir():
        impact_name = impact_dir.name
        print(f"{agg_type}-{impact_name}...")
        cur_config = {
            "agg_type": agg_type,
            "impact_type": impact_name,
        }
        ### FULL
        for model_dir in impact_dir.iterdir():
            model_name = model_dir.name
            print(f"\t{model_name}...")
            ## ImpactMe LR models
            if agg_type in ["max", "avg", "block"]:
                for seg_dir in model_dir.iterdir():
                    seg_type = seg_dir.name
                    print(f"\t\t{seg_type}...")
                    cur_config.update(
                        {
                            "model_name": model_name,
                            "seg_type": seg_type,
                            "desc_type": "-----",
                        }
                    )
                    cur_config["seg_type"] = seg_type
                    best_params_score_dict = get_best_params_score(
                        seg_dir,
                        show_progress=True,
                        optimize=OPTIMIZE,
                    )
                    new_row = {
                        **cur_config,
                        **best_params_score_dict,
                    }
                    df_rows.append(new_row)
            ## 0-shot
            else:
                for desc_dir in model_dir.iterdir():
                    desc_type = desc_dir.name.split("_")[1]
                    print(f"\t\t{desc_type}...")
                    cur_config.update(
                        {
                            "model_name": model_name,
                            "seg_type": "-----",
                            "desc_type": desc_type,
                        }
                    )
                    best_params_score_dict = get_best_params_score(
                        desc_dir,
                        show_progress=True,
                        optimize=OPTIMIZE,
                    )
                    new_row = {
                        **cur_config,
                        **best_params_score_dict,
                    }
                    df_rows.append(new_row)

full_result_df = pd.DataFrame(df_rows)
display(full_result_df)

Format

In [ ]:
other = "r2" if OPTIMIZE == "mse" else "mse"
ascend_list = (
    [True, True, True, False] if OPTIMIZE == "mse" else [True, True, False, True]
)
reorder_cols = [
    "agg_type",
    "impact_type",
    f"avg_{OPTIMIZE}",
    f"stdev_{OPTIMIZE}",
    f"avg_{other}",
    f"stdev_{other}",
    "l1",
    "alpha",
    "model_name",
    "seg_type",
    "desc_type",
]
full_result_df_ordered = full_result_df[reorder_cols]
full_result_df_ordered.sort_values(
    by=[
        "agg_type",
        "impact_type",
        f"avg_{OPTIMIZE}",
        f"avg_{other}",
    ],
    ascending=ascend_list,
    na_position="last",
    inplace=True,
)
full_result_df_ordered

## Export 

In [ ]:
export_dir = BASE_PATH / "initial_results" / f"params_optimize_{OPTIMIZE}"
print(f"Exporting to: {export_dir}")
## Full
full_path = export_dir / "full.tsv"
if full_path.exists():
    full_path.unlink()
full_path.parent.mkdir(exist_ok=True, parents=True)
full_result_df_ordered.to_csv(full_path, sep="\t")